# មេរៀន 04 - លំនាំរចនាសម្រាប់ការប្រើឧបករណ៍

In this lesson you will learn the **ការប្រើឧបករណ៍** design pattern for AI agents using the Microsoft Agent Framework (Python). We cover:

- កំណត់ឧបករណ៍មុខងារ ជាមួយ `@tool` decorator និងប៉ារ៉ាម៉ែត្រមានប្រភេទ
- ផ្តល់ស្កីម៉ាសសម្រាប់ឧបករណ៍ ដើម្បីឲ្យម៉ូដែលដឹងថា​ឧបករណ៍នីមួយៗ​ធ្វើអ្វី
- គ្រប់គ្រងការប្រតិបត្តិឧបករណ៍ដោយប្រើ `approval_mode`
- បញ្ចូន **ទិន្នន័យដែលមានរចនាសម្ព័ន្ធ** តាមរយៈម៉ូដែល Pydantic និង `response_format`

The scenario is a **ភ្នាក់ងារកក់ដំណើរ** that can look up destinations, check availability, and retrieve flight information.


## ការតំឡើង


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ការកំណត់ឧបករណ៍ជាមួយ @tool Decorator

`@tool` decorator បម្លែងអនុគមន៍ Python ធម្មតា ទៅជា​ឧបករណ៍មួយដែលភ្នាក់ងារអាចហៅបាន។  
ចំណុចសំខាន់ៗ៖

- **docstring** នឹងក្លាយជា​ពិពណ៌នាឧបករណ៍​ដែលម៉ូឌែលមើលឃើញ។  
- **Type annotations** (រួមទាំង `Annotated` ជាមួយការពិពណ៌នា) កំណត់រចនាសម្ព័ន្ធឧបករណ៍។  
- `approval_mode` គ្រប់គ្រងថាតើអ្នកប្រើត្រូវអនុម័តការហៅរាល់ម្តងមុនវាអនុវត្តឬទេ។


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ការបង្កើត Agent ជាមួយឧបករណ៍ច្រើន

ផ្ញើឧបករណ៍ទាំងបីទៅឲ្យ client ដូច្នេះ model អាចហៅឧបករណ៍ណាមួយដែលវាត្រូវការ ដើម្បីឆ្លើយសំណួររបស់ user។


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## លទ្ធផលដែលមានរចនាសម្ព័ន្ធជាមួយឧបករណ៍

ដោយកំណត់ `response_format` ទៅជា ម៉ូដែល Pydantic, ភ្នាក់ងារ ត្រូវបានបង្ខំឱ្យត្រឡប់មកជា វត្ថុ JSON ដែលមានប្រភេទច្បាស់ ជំនួសអត្ថបទទ្រង់ទ្រាយសេរី។ វាមានប្រយោជន៍នៅពេលដែលកូដដែលដំណើរការបន្ត ត្រូវការប្រើលទ្ធផលដោយរបៀបកម្មវិធី។


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## លំនាំការអនុម័តឧបករណ៍

ប៉ារ៉ាម៉ែត្រ `approval_mode` លើ `@tool` កំណត់ថា ការហៅឧបករណ៍ត្រូវការការអនុញ្ញាតពីមនុស្សមុនពេលអនុវត្តឬអត់:

| ម៉ូដ | អាកប្បកិរិយា |
|---|---|
| `"never_require"` | ឧបករណ៍រត់ដោយស្វ័យប្រវត្តិ — មិនចាំបាច់មានការបញ្ជាក់ពីអ្នកប្រើទេ។ |
| `"always_require"` | គ្រប់ការហៅត្រូវការការអនុម័តពីអ្នកប្រើ មុនពេលវា​អនុវត្ត។ |

ប្រើ `"always_require"` សម្រាប់ឧបករណ៍ដែលមានផលប៉ះពាល់ខាងក្រោយ (ឧ. ការកក់សំបុត្រយន្តហោះ, ការទូទាត់តាមកាតឥណទាន) ដើម្បីឲ្យមនុស្សនៅក្នុងដំណើរការ។


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## សង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀប៖

1. **កំណត់ឧបករណ៍** ដោយប្រើ `@tool` decorator ជាមួយប៉ារ៉ាម៉ែត្រប្រភេទ និង docstrings ដែលបម្រើជាស្គីម៉ាដើម្បីកំណត់រចនាសម្ព័ន្ធឧបករណ៍។
2. **រៀបចំឧបករណ៍ច្រើន** ដើម្បីឱ្យភ្នាក់ងារអាចហៅពួកវាទៅតាមលំដាប់ដើម្បីឆ្លើយសំណួរដែលស្មុគស្មាញ។
3. **ត្រឡប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ** ដោយផ្ទេរម៉ូដែល Pydantic ជា `response_format`។
4. **គ្រប់គ្រងការអនុម័តឧបករណ៍** ជាមួយ `approval_mode` ដើម្បីរក្សាមនុស្សឱ្យនៅក្នុងដំណើរការសម្រាប់ប្រតិបត្តិការមានហានិភ័យ។

លំនាំទាំងនេះបង្កើតមូលដ្ឋានសម្រាប់សាងសង់ភ្នាក់ងារដែលទុកចិត្តបាន និងរួចសម្រាប់ផលិតកម្ម ដែលអាចអន្តរកម្មជាមួយប្រព័ន្ធខាងក្រៅបានយ៉ាងសុវត្ថិភាព។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែដោយបច្ចេកវិទ្យា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយ៉ាងណា ខណៈដែលយើងខិតខំរក្សាការត្រឹមត្រូវ សូមយល់ឲ្យបានច្បាស់ថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាដើមគួរត្រូវបានទុកជាប្រភពដែលអាចអផ្ចេីយជានិយមសម្រាប់យោង។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឱ្យប្រើការបកប្រែដោយមនុស្សដែលមានវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
